In [1]:
import sys
sys.path.insert(0, '..')

caminho_poemas_portinari = '../Data/poemas.csv'
caminho_poemas_extra = '../Data/portuguese-poems.csv'
caminho_noticias = '../Data/Historico_de_materias.csv'

In [2]:
import pandas as pd

portinari = pd.read_csv(caminho_poemas_portinari)[['estrofes']].rename(columns={'estrofes': 'texto'})
noticias  = pd.read_csv(caminho_noticias)[['conteudo_noticia']].rename(columns={'conteudo_noticia': 'texto'}).sample(n=111, random_state=42)
poemas    = pd.read_csv(caminho_poemas_extra)[['Content']].rename(columns={'Content': 'texto'}).sample(n=111, random_state=42)

portinari['label'] = 1
noticias['label']  = 0
poemas['label']    = 0

df_final = pd.concat([portinari, noticias, poemas]).sample(frac=1, random_state=42).reset_index(drop=True)

In [3]:
#Apenas para testar rapido
df_final = df_final.sample(n=60, random_state=42)

In [ ]:
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

MODEL = 'neuralmind/bert-base-portuguese-cased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL)

train, val = train_test_split(df_final, test_size=0.2, stratify=df_final['label'])

def tokenize(batch):
    return tokenizer(batch['texto'], truncation=True, padding='max_length', max_length=512)

ds_train = Dataset.from_pandas(train).map(tokenize, batched=True)
ds_val   = Dataset.from_pandas(val).map(tokenize, batched=True)

model = BertForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir='./bertimbau-portinari',
    num_train_epochs=2,# antes 4
    per_device_train_batch_size=4,# antes 8
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val)
trainer.train()

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [ ]:
from transformers import pipeline

clf = pipeline('text-classification', model='./bertimbau-portinari', tokenizer=tokenizer)
clf("Minha terra tem palmeiras onde canta o sabiá...")